# ViPRA — Qwen2.5-3B-Instruct, train đầy đủ và validation trên dev

Backbone **3,09B**, LoRA FP16 + **FSDP2 song song trên T4 x2**. Bật Internet và Add Input chứa hai CSV.
Clone nhánh `Nhotest` vào `/kaggle/working/VLSP`:

```bash
git clone --branch Nhotest --single-branch https://github.com/NgocNhoNguyen2605n/VLSP.git /kaggle/working/VLSP
```

Học toàn bộ train, dùng dev validation sau mỗi epoch. Chọn best epoch theo đáp án + đơn vị,
xuất adapter của checkpoint thắng và dùng trực tiếp. Không chia 90/10, không retrain.


In [ ]:
import os  # Thiết lập môi trường cho các tiến trình train/evaluate.
import subprocess  # Chạy Python riêng để trả lại VRAM sau mỗi tác vụ.
import sys  # Dùng đúng Python của notebook Kaggle.
from pathlib import Path  # Đường dẫn repo, Kaggle Input và kết quả.

REPO = Path("/kaggle/working/VLSP")
DATA_ROOT = "/kaggle/input"
RUN_DIR = "/kaggle/working/vipra_qwen25_3b_5epochs"  # Chọn thư mục mới cho mỗi experiment.
assert (REPO / "requirements.txt").is_file(), "Hãy clone repo vào /kaggle/working/VLSP trước."

os.environ.update(PYTHONHASHSEED="42", CUBLAS_WORKSPACE_CONFIG=":4096:8",
                  TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_PROGRESS_BARS="1",
                  HF_HOME="/tmp/vipra_hf_cache")  # Không lưu backbone vào output notebook.

def run(module, *options):
    "Chạy tác vụ trong repo, dừng notebook nếu có lỗi."
    subprocess.run([sys.executable, "-m", module, *options], cwd=REPO, check=True)

# Gỡ torchao cũ của Kaggle: PEFT kiểm tra cả khi chỉ dùng LoRA FP16.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y",
                "torchao"], check=True)

# Cài dependency; cần PyTorch CUDA >= 2.8 cho FSDP2.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO / "requirements.txt")], check=True)


## 1. Kiểm tra dữ liệu và template

Tự tìm CSV trong Input. Nếu có nhiều bản cùng tên, thu hẹp `DATA_ROOT`
hoặc truyền `--train-file` và `--dev-file` cụ thể vào cả check/train.
Không tải trọng số backbone trong bước này.


In [ ]:
# Kiểm tra các mẫu và mask assistant; mẫu dài sẽ báo lỗi thay vì bị cắt.
run("baseline.check", "--data-root", DATA_ROOT)


## 2. Train 5 epoch và chọn best epoch trên dev

**676 mẫu train hợp lệ / 339 mẫu dev validation**, seed 42. Train dùng toàn bộ train_dataset.csv;
LD111 thiếu answer được bỏ theo cấu hình. Dev chỉ dùng chấm loss và sinh đáp án, không cập nhật gradient.

Sau mỗi epoch, validation greedy theo batch 4, tối đa 300 token/câu, lưu checkpoint + predictions + metrics.
Chọn content_tolerant accuracy cao nhất; nếu hòa: exact, loss thấp hơn, epoch sớm hơn.
Kết thúc 5 epoch, xuất đúng checkpoint thắng vào **RUN_DIR/adapter**, không retrain.

Batch train 4/GPU × 2 GPU, tích lũy 2 = effective batch 16. Có **85 batch train/epoch** và
**85 batch generation validation/epoch**. Max_length 2048, LoRA r=16/alpha=32, backbone FP16.

Launcher dùng torchrun 2 worker FSDP2. Hai rank cùng sinh một batch validation, chỉ rank 0 ghi điểm.
Decoder frozen shard; embedding/head tied và LoRA replicate. Gradient LoRA all-reduce sau backward;
activation checkpoint bên trong FSDP, backward prefetch tắt. Preflight CUDA/NCCL phải PASS trước khi nạp 3B.

Tiến độ hiển thị batch và ETA của pha hiện tại. Với output pipe/Save Version, mỗi bản cập nhật
có xuống dòng: mỗi 10 batch hoặc sau 30 giây ở batch vừa hoàn tất, luôn in batch đầu/cuối.
ETA không tính validation hay checkpoint sắp tới. verbose_logging=false ẩn bảng thông tin chi tiết.

Chọn RUN_DIR mới mỗi lần chạy. Nếu NCCL abort, restart session trước khi chạy lại.
rank-0-progress.json và rank-1-progress.json trong RUN_DIR/search ghi pha cuối của mỗi worker.


In [ ]:
# Train toàn bộ train -> validation dev mỗi epoch -> xuất best adapter, không retrain.
run("baseline.experiment", "--data-root", DATA_ROOT, "--output-dir", RUN_DIR)


## 3. Xem best epoch và điểm dev đã lưu

Adapter dùng trực tiếp: RUN_DIR/adapter. Không cần nạp lại model hoặc sinh lại validation.
Dev đã dùng chọn checkpoint, nên điểm này là validation, không phải test độc lập.


In [ ]:
import json

selection = json.loads((Path(RUN_DIR) / "selection.json").read_text())
best = selection["best"]
print(f"Best epoch: {best['epoch']} | Đáp án + đơn vị đúng: {best['content_tolerant']}/{best['total']}")
print("Adapter:", str(Path(RUN_DIR) / "adapter"))
print("Predictions dev:", str(Path(RUN_DIR) / "search" / "validation" / f"epoch-{best['epoch']}" / "predictions.jsonl"))


## 4. Đánh giá lại best adapter (tùy chọn)

Mặc định bỏ qua để tiết kiệm thời gian vì dev đã được chấm sau mỗi epoch.
Nếu cần đo thời gian từng câu/P1 ước lượng, bật RUN_EXTRA_EVALUATION.
Lệnh evaluate dùng RUN_DIR/adapter của best epoch và sinh một câu/lượt; không retrain.


In [ ]:
RUN_EXTRA_EVALUATION = False
if RUN_EXTRA_EVALUATION:
    run("baseline.evaluate", "--run-dir", RUN_DIR)
else:
    print("Dùng điểm dev của best epoch đã lưu; bỏ qua đánh giá lại.")


Manifest CSV: RUN_DIR/split.json. Xếp hạng: RUN_DIR/selection.json.
Run training: RUN_DIR/search/. Adapter **best epoch**: RUN_DIR/adapter/.
Điểm dev từng epoch: RUN_DIR/search/validation/epoch-N/metrics.json và predictions.jsonl.
Lệnh evaluate tùy chọn lưu RUN_DIR/evaluation/; muốn chạy lại dùng --result-dir khác.
